In [40]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time
import csv
import re

In [41]:
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://yandex.ru/maps/?display-text=%D0%9F%D0%B0%D0%BD%D1%81%D0%B8%D0%BE%D0%BD%D0%B0%D1%82%20%D0%B4%D0%BB%D1%8F%20%D0%BF%D0%BE%D0%B6%D0%B8%D0%BB%D1%8B%D1%85%20%D0%BB%D1%8E%D0%B4%D0%B5%D0%B9%2C%20%D0%BF%D1%80%D0%B5%D1%81%D1%82%D0%B0%D1%80%D0%B5%D0%BB%D1%8B%D1%85%20%D0%B8%20%D0%B8%D0%BD%D0%B2%D0%B0%D0%BB%D0%B8%D0%B4%D0%BE%D0%B2&ll=37.658051%2C55.598923&mode=search&sctx=ZAAAAAgBEAAaKAoSCZm36jpU2UJAEUIj2Lj%2B2UtAEhIJA15m2CjrAEARz4b8M4N45z8iBgABAgMEBSgKOABAAUgBagJydZ0BzczMPaABAKgBAL0BRUbtU8IBjgHTzfi3%2FwOQzMXi1ALusfDYgAK8uYqM9gP0mpn%2FA8rWtLUFy5rDrbMFp96y3gPXtdKIBvaGhbL9ApnY8YGMA9m59%2BnMAfrq5eMstufemJwFgOW9gPUCp4LszJoDloSX0rEG2LyhhNoDutXj3AbNy6zlBfKCwu2kBdudhsKdBp3Pjf62A5CrlZWqBMOFwfw3ggIbKChjYXRlZ29yeV9pZDooMTg0MTA2MzA4KSkpigIJMTg0MTA2MzA4kgIAmgIMZGVza3RvcC1tYXBzqgLvATIwODYzNDI0MjYzMywyMDY5ODgzODMwNTIsNTI2MzE4NTcwNDMsMTQwMzI3OTAxODMzLDIwODYxOTM3MzgyOCwxMTY5MDExMTM4MTUsODU2ODUyODQ0MzQsMTYzNzI0OTc2MTQ0LDE4ODg1MTAwNzI3NiwxOTAzNTk4NjcyNCwxMjI1NjExMjYyMTYsMTU2NzgyNzkyMTY4LDE2MTQ3NzI2MzU1NiwxNjUxMDMzNzA4MzksMzExNzIxMjEzNjksNjI1MzE2MjQ0MzUsMTYzODYwMTAyNDY4LDEzMTk4OTEyOTA3MCw4ODkxNDgzNjgy2gIoChIJWZpbIazgQkARKwyFVifHS0ASEgkwbXGNz6QJQBFgBmL62trxP%2BACAQ%3D%3D&sll=37.658051%2C55.598923&sspn=2.790527%2C0.970402&text=%7B%22text%22%3A%22%D0%9F%D0%B0%D0%BD%D1%81%D0%B8%D0%BE%D0%BD%D0%B0%D1%82%20%D0%B4%D0%BB%D1%8F%20%D0%BF%D0%BE%D0%B6%D0%B8%D0%BB%D1%8B%D1%85%20%D0%BB%D1%8E%D0%B4%D0%B5%D0%B9%2C%20%D0%BF%D1%80%D0%B5%D1%81%D1%82%D0%B0%D1%80%D0%B5%D0%BB%D1%8B%D1%85%20%D0%B8%20%D0%B8%D0%BD%D0%B2%D0%B0%D0%BB%D0%B8%D0%B4%D0%BE%D0%B2%22%2C%22what%22%3A%5B%7B%22attr_name%22%3A%22category_id%22%2C%22attr_values%22%3A%5B%22184106308%22%5D%7D%5D%7D&z=9"
driver.get(url)
time.sleep(5)

scroll_container = driver.find_element(By.CSS_SELECTOR, "div.scroll__container")

prev_count = 0
no_change_count = 0
for i in range(40):
    driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
    time.sleep(2)
    
    cards = driver.find_elements(By.CSS_SELECTOR, "li.search-snippet-view")
    current_count = len(cards)
    print(f"   Шаг {i+1}: загружено {current_count} организаций")
    
    if current_count == prev_count:
        no_change_count += 1
        if no_change_count >= 5:
            print("   ⏸️ Достигнут конец списка")
            break
    else:
        no_change_count = 0
        prev_count = current_count

cards = driver.find_elements(By.CSS_SELECTOR, "li.search-snippet-view")
results = []
seen_names = set()

for card in cards:
    try:
        # Название
        name = ""
        try:
            link = card.find_element(By.CSS_SELECTOR, "a[class*='link-overlay']")
            name = link.get_attribute("aria-label")
        except:
            pass
        
        if not name:
            try:
                title = card.find_element(By.CSS_SELECTOR, "div[class*='title']")
                name = title.text.strip()
            except:
                continue
        
        if not name or len(name) < 3 or name in seen_names:
            continue
        seen_names.add(name)
        
        # Адрес
        address = "Не указан"
        try:
            addr = card.find_element(By.CSS_SELECTOR, "[class*='address']")
            address = addr.text.strip()
        except:
            pass
        
        # Рейтинг
        rating = ""
        try:
            rating_elem = card.find_element(By.CSS_SELECTOR, "[class*='rating-text']")
            rating = rating_elem.text.strip()
        except:
            pass
        
        results.append({
            "название": name,
            "адрес": address,
            "рейтинг": rating
        })
        print(f"   ✅ {name[:50]}")
        
    except Exception as e:
        continue

driver.quit()


with open('пансионаты.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['название', 'адрес', 'рейтинг'])
    writer.writeheader()
    writer.writerows(results)

print(f"\n🎉 ИТОГО СОБРАНО: {len(results)} пансионатов!")


   Шаг 1: загружено 12 организаций
   Шаг 2: загружено 19 организаций
   Шаг 3: загружено 25 организаций
   Шаг 4: загружено 32 организаций
   Шаг 5: загружено 39 организаций
   Шаг 6: загружено 46 организаций
   Шаг 7: загружено 53 организаций
   Шаг 8: загружено 61 организаций
   Шаг 9: загружено 66 организаций
   Шаг 10: загружено 74 организаций
   Шаг 11: загружено 82 организаций
   Шаг 12: загружено 89 организаций
   Шаг 13: загружено 89 организаций
   Шаг 14: загружено 89 организаций
   Шаг 15: загружено 89 организаций
   Шаг 16: загружено 89 организаций
   Шаг 17: загружено 89 организаций
   ⏸️ Достигнут конец списка
   ✅ Дружный Берег
   ✅ Ника
   ✅ Доброта
   ✅ Теплые беседы
   ✅ Климовский дом-интернат для престарелых и инвалидо
   ✅ Океан милосердия
   ✅ Лайф Пенсион
   ✅ Пансионат для пожилых Надежда
   ✅ Оазис
   ✅ Золотая пора
   ✅ Добро
   ✅ Серебряные зори
   ✅ Академия Долголетия
   ✅ Пансионат Вера
   ✅ Пансионат для пожилых людей Щелково
   ✅ Пансионат Плюс
   ✅ Радо